In [1]:
import numpy as np
import dask , dask.distributed
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy
import cmocean.cm as cmo
import warnings
warnings.simplefilter('ignore')
import dask_jobqueue
from astropy.convolution import Box2DKernel, convolve
import pandas as pd

In this file, the data was merged into a single file for more practicability later

# monthly

In [ ]:
# windstress_1PctTo2X.nc
# windstress_ctrl.nc

In [ ]:
#SST, SSH, dic_stf, o2_stf, sens_heat, evap_heat, mld, jp_all =jp_uptake-remin-recycle
#tau_x, tau_y, wind_stress

In [2]:
path_to='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5_concat/0181-0190/'
path_from_ctrl='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5/ctrl/'
path_from_1PctTo2X='/gxfs_work/geomar/smomw577/mesoscale_eddies/MOM5/1PctTo2X/'

In [ ]:
def merging(path):
    ice_daily=xr.open_mfdataset(path+'01*0101.ice_daily.nc')[['SSH', 'SST']]#yt, xt
    bio=xr.open_mfdataset(path+'01*0101.ocean_minibling_100m.nc')[['jp_uptake', 'jp_reminp', 'jp_recycle']] #yt_ocean, xt_ocean
    hf=xr.open_mfdataset(path+'01*0101.ocean_bdy_flux.nc')[['sens_heat', 'evap_heat']] #yt_ocean, xt_ocean
    gas_flux=xr.open_mfdataset(path+'01*0101.ocean_minibling_term_src.nc')[['o2_stf', 'dic_stf']] *60*60*24*365 #yt_ocean, xt_ocean
    mld=xr.open_mfdataset(path+'01*0101.ocean.nc')[['mld']] #yt_ocean, xt_ocean
    
    ice_daily=ice_daily.interp_like(bio.jp_uptake)
    ds_out = xr.merge([
    ice_daily,
    bio,
    hf,
    gas_flux,
    mld
    ])
    
    ds_out['jp_all'] = bio.jp_uptake - bio.jp_reminp - bio.jp_recycle
    
    ds_out = ds_out.drop_vars(['jp_uptake', 'jp_reminp', 'jp_recycle'])
    if 'ctrl' in path:
        ds_out.to_netcdf(path_to + 'MOM5_control_monthly_0181-0190_all.nc')
    elif '1PctTo2X' in path:
        ds_out.to_netcdf(path_to + 'MOM5_1PctTo2X_monthly_0181-0190_all.nc.nc')

In [ ]:
merging(path_from_ctrl)
merging(path_from_1PctTo2X)

## wind

files windstress_ctrl.nc and windstress_1PctTo2X.nc were created with cdo and contain only tau_x and tau_y

# adjust metadata and files for publication

In [3]:
path='/gxfs_work/geomar/smomw577/mesoscale_eddies/BOX_filtered/0181-0190/'
ds1=xr.open_dataset(path+'3x3box_median_anomaly_monthly_0181-0190_all.nc')[['o2_stf','o2_stf_1PctTo2X', 'dic_stf', 'dic_stf_1PctTo2X','SSH', 'SSH_1PctTo2X', 'SST', 'SST_1PctTo2X', 'sens_heat', 'sens_heat_1PctTo2X', 'evap_heat', 'evap_heat_1PctTo2X', 'jp_all', 'jp_all_1PctTo2X']]
ds2=xr.open_dataset(path+'categorization_simple.nc')
ds3=xr.open_dataset(path+'categorization_simple_1PctTo2X.nc')
ds4=xr.open_dataset(path+'3x3box_median_corr_monthly_co2-o2_0181-0190_noice_all.nc')
ds5=xr.open_dataset(path+'3x3box_median_corr_monthly_jp_0181-0190_noice_all.nc')
grid=xr.open_dataset(path+'ocean_grid.nc')

In [4]:
ds1.o2_stf.attrs = {
    "standard_name": "surface_downward_mole_flux_of_molecular_oxygen",
    "long_name": "'Downward' indicates a vector component which is positive when directed downward (negative upward). In accordance with common usage in geophysical disciplines, 'flux' implies per unit area, called 'flux density' in physics. The surface called 'surface' means the lower boundary of the atmosphere.",
    "units": 'mol m-2 yr-1'
}
ds1.o2_stf_1PctTo2X.attrs = {
    "standard_name": "surface_downward_mole_flux_of_molecular_oxygen",
    "long_name": "'Downward' indicates a vector component which is positive when directed downward (negative upward). In accordance with common usage in geophysical disciplines, 'flux' implies per unit area, called 'flux density' in physics. The surface called 'surface' means the lower boundary of the atmosphere.",
    "units": 'mol m-2 yr-1'
}

ds1.dic_stf.attrs = {
    "standard_name": "surface_downward_mole_flux_of_carbon_dioxide",
    "long_name": "'Downward' indicates a vector component which is positive when directed downward (negative upward). In accordance with common usage in geophysical disciplines, 'flux' implies per unit area, called 'flux density' in physics. The surface called 'surface' means the lower boundary of the atmosphere. The chemical formula for carbon dioxide is CO2. ",
    "units": 'mol m-2 yr-1'
}

ds1.dic_stf_1PctTo2X.attrs = {
    "standard_name": "surface_downward_mole_flux_of_carbon_dioxide",
    "long_name": "'Downward' indicates a vector component which is positive when directed downward (negative upward). In accordance with common usage in geophysical disciplines, 'flux' implies per unit area, called 'flux density' in physics. The surface called 'surface' means the lower boundary of the atmosphere. The chemical formula for carbon dioxide is CO2. ",
    "units": 'mol m-2 yr-1'
}

ds1.SSH.attrs = {
    "long_name" : "sea surface height",
    "units": "m"

}

ds1.SSH_1PctTo2X.attrs = {
    "long_name": "sea surface height",
    "units": "m"
}

ds1.SST.attrs = {
    "standard_name": "sea_surface_temperature",
    "units": "deg-C"

}

ds1.SST_1PctTo2X.attrs = {
    "standard_name": "sea_surface_temperature",
    "units": "deg-C"
}

ds1.sens_heat.attrs = {
    "standard_name": "surface_downward_sensible_heat_flux",
    "units": "W m-2"

}

ds1.sens_heat_1PctTo2X.attrs = {
    "standard_name": "surface_downward_sensible_heat_flux",
    "units": "W m-2"
}

ds1.evap_heat.attrs = {
    "standard_name": "surface_downward_latent_heat_flux",
    "units": "W m-2"

}

ds1.evap_heat_1PctTo2X.attrs = {
    "standard_name": "surface_downward_latent_heat_flux",
    "units": "W m-2"
}

#ds1.mld.attrs = {
#    "standard_name": "ocean_mixed_layer_thickness_defined_by_sigma_t",
#    "units": "m"
#
#}
#
#ds1.mld_1PctTo2X.attrs = {
#    "standard_name": "ocean_mixed_layer_thickness_defined_by_sigma_t",
#    "units": "m"
#}

ds1.jp_all.attrs = {
    "long_name": "net community production",
    "units": "mol m-2 s-1"

}

ds1.jp_all_1PctTo2X.attrs = {
    "long_name": "net community production",
    "units": "mol m-2 s-1"
}


In [5]:
###to be deleted
ds2=ds2.rename({"corr_o2_stf_dic_stf": "categorization"})
ds3=ds3.rename({"corr_o2_stf_dic_stf_1PctTo2X": "categorization"})


In [6]:
ds2.categorization.attrs = {
    "long_name": "Categorization of mesoscale CO2/O2 Flux drivers based on mesoscale correlation between co2/o2 and co2/jp. 1 is 'solubility-driven', 2 is 'productivity-driven', 3 is 'respiration-driven'. For more information read the respective manuscript.",
    "units": "None"
}

In [7]:
ds3.categorization.attrs = {
    "long_name": "Categorization of mesoscale CO2/O2 Flux drivers based on mesoscale correlation between co2/o2 and co2/jp. 1 is 'solubility-driven', 2 is 'productivity-driven', 3 is 'respiration-driven'. For more information read the respective manuscript.",
    "units": "None"
}

In [8]:
ds4.corr_o2_stf_dic_stf.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale CO2 and O2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

ds4.corr_o2_stf_dic_stf_1PctTo2X.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale CO2 and O2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

In [9]:
ds5.corr_jp_dic_stf_ctrl.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale net community production and CO2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

ds5.corr_jp_dic_stf_1PctTo2X.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale net community production and CO2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

ds5.corr_jp_o2_stf_ctrl.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale net community production and O2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

ds5.corr_jp_o2_stf_1PctTo2X.attrs = {
    "long_name": "Pearson correlation r value of correlation between mesoscale net community production and O2 Flux anomalies along time-dimension (monthly).",
    "units": "None"
}

In [10]:
grid=grid[['area_t']]

In [11]:
datasets=[ds1,ds2,ds3,ds4,ds5, grid]
titles=["Box-Filtered Mesoscale Anomalies GFDL CM2.6",
       "Mesoscale CO2/O2 Flux Driver Categorization Control Run",
       "Mesoscale CO2/O2 Flux Driver Categorization 1PctTo2X Run",
       "Pearson Correlation between mesoscale CO2 and O2 Flux anomalies",
       "Pearson Correlation between mesoscale CO2/O2 Flux anomalies and net community production",
        "Ocean Grid Area"
       ]

In [12]:
for i,ds in enumerate(datasets):
    ds.attrs.update(
    {"date_created":'2026-06-04',                                 # when created
     "date_issued":'2026-06-04',                                  # date when data were formally issued     
     "date_metadata_modified":'2026-06-04',                       # or use `date -I` to select today's date with bash script   
     "creator_name":'Nana Hocke',   
     "creator_email":'nana.hocke@gmail.com',    
     "creator_id":'https://orcid.org/0009-0003-1746-5912',   
     "creator_affiliation":'GEOMAR Helmholtz Centre for Ocean Research Kiel',
     "creator_affiliation_id":'https://ror.org/02h2x0161',
     "title":titles[i],    # title of data represented in file/dataset   
     "project":'',   
     "contributor_name":'Ivy Frenger',               # relevant persons involved or contributed creating these data!   
     "contributor_email":'ifrenger@geomar.de',    
     "contributor_id":'https://orcid.org/0000-0002-3490-7239',
     "institution":'GEOMAR Helmholtz Centre for Ocean Research Kiel',  
     "institution_id":'https://ror.org/02h2x0161',                # affiliation of creator/primary author
     "research_division":'Marine Biogeochemistry',# affiliation of creator/primary author       
     "research_unit":'Biogeochemical Modelling',                            # affiliation of creator/primary author
     "license":'CC BY 4.0  (https://creativecommons.org/licenses/by/4.0/deed.en)',   
     "naming_authority":'de.geomar',    
     "publisher_name":'GEOMAR Helmholtz Centre for Ocean Research Kiel',   
     "publisher_email":'datamanagement@geomar.de',   
     "processing_level":'Level 4 (numerical simulation output)',  # can be used if derived from simulation output without calibration and error processing chain   
     "featureType":'point',                                       # see table here: https://cfconventions.org/cf-conventions/cf-conventions.html#_features_and_feature_types
     "cdm_data_type":'Grid',                                # E.g., "Grid", "Image", "Station", "Trajectory", "Radial", see:  
     "handle":'',                                                 # this can be added by datamanagement after handle is registered if not known
     "references":'',                                             # if none leave empty; Published or web-based references that describe the data or methods used to produce it. URL or DOI recommended.
     "keywords":'CARBON DIOXIDE, OXYGEN, EDDIES, GREENHOUSE GAS FLUX, MESOSCALE EDDIES',                                 # please search here: https://gcmd.earthdata.nasa.gov/KeywordViewer
     # for example keywords such as: OCEAN GENERAL CIRCULATION MODELS/(OGCM), REGIONAL OCEAN MODELS,  CLIMATE INDICATORS, ATMOSPHERIC/OCEAN INDICATORS, ATLANTIC OCEAN, WATER TEMPERATURE, ...
     "keywords_vocabulary":'GCMD:GCMD Keywords',                  # which keywords_vocabulary you followed
     "acknowledgement":'The authors thank the GFDL community for the tremendous investment of computational resources and manpower required to carry out the CM2.6 control simulations with a nominal lateral ocean resolution of 0.1 of a degree. ',                # example to acknowledge the funding for resources for this work could also be project funding acknowledgement
     "comment":'We here archive postprocessed GFDL climate model (CM2.6) data. The raw data, CM2.6 simulations, are further described in earlier publications, including in Delworth et al, 2012, doi: 10.1175/JCLI-D-11-00316.1; Griffies et al, 2015, doi: 10.1175/JCLI-D-14-00353.1; Galbraith et al, 2015, doi: 10.1002/2015MS000463',                                                # Miscellaneous information about the data or methods used to produce it.
     "realm":'ocean',                                             # otherwise leave empty; from CMIP6 realms such as 'ocean', 'ocnBgchem', 'seaIce', ... 
                                                                 # more in direct connection of model output; see https://github.com/WCRP-CMIP/CMIP6_CVs/blob/main/CMIP6_realm.json
     "source":'GFDL CM2.6',                                      # The method of production of the original data. 
                                                                 # If it was model-generated, source should name the model and its version. 
                                                                 # If it is observational, source should characterize it. This attribute is defined in the CF Conventions. 
                                                                 # Examples: 'temperature from CTD #1234'; 'world_model_v.0.1'.
     "Conventions":'CF-1.6' })                                   # which convention/schema do the metadata follow here

In [13]:
filename=[
    '3x3box_median_anomaly_monthly_0181-0190_all.nc',
    'categorization_simple.nc',
    'categorization_simple_1PctTo2X.nc',
    '3x3box_median_corr_monthly_co2-o2_0181-0190_noice_all.nc',
    '3x3box_median_corr_monthly_jp_0181-0190_noice_all.nc',
    'ocean_grid.nc'
]
for i,ds in enumerate(datasets):
    ds.to_netcdf('/gxfs_work/geomar/smomw577/mesoscale_eddies/submitted/'+filename[i])